In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Teknik mitigasi dan penindasan ralat

> **Note:** Keluaran beta model pelaksanaan baharu kini tersedia. Model pelaksanaan terarah menyediakan lebih banyak fleksibiliti apabila menyesuaikan aliran kerja mitigasi ralat anda. Lihat panduan [Model pelaksanaan terarah](/guides/directed-execution-model) untuk maklumat lanjut.


<details>
<summary><b>Package versions</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Teknik mitigasi ralat dan penindasan ralat digunakan untuk meningkatkan kualiti keputusan apabila menskalakan kepada beban kerja yang lebih besar. Halaman ini menyediakan penjelasan tahap tinggi tentang teknik penindasan ralat dan mitigasi ralat yang tersedia melalui Qiskit Runtime.

Sel berikut mengimport primitif Estimator dan membuat backend yang akan digunakan untuk memulakan Estimator dalam sel kod kemudian.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Dynamical decoupling
Litar kuantum dilaksanakan pada perkakasan IBM&reg; sebagai urutan nadi gelombang mikro yang perlu dijadualkan dan dijalankan pada selang masa yang tepat.
Malangnya, interaksi tidak diingini antara qubit boleh membawa kepada ralat koheren pada qubit yang melahu. Dynamical decoupling berfungsi dengan memasukkan urutan nadi pada qubit yang melahu untuk kira-kira membatalkan kesan ralat ini. Setiap urutan nadi yang dimasukkan setara dengan operasi identiti, tetapi kehadiran fizikal nadi tersebut mempunyai kesan menindas ralat.
Terdapat banyak pilihan urutan nadi yang mungkin, dan urutan mana yang lebih baik untuk setiap kes tertentu masih merupakan [bidang penyelidikan aktif](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Perlu diingat bahawa dynamical decoupling terutamanya berguna untuk litar yang mengandungi jurang di mana sebahagian qubit duduk melahu tanpa sebarang operasi yang bertindak ke atasnya. Jika operasi dalam litar sangat padat, sedemikian rupa sehingga semua qubit sibuk sebahagian besar masa, maka penambahan nadi dynamical decoupling mungkin tidak meningkatkan prestasi. Malah, ia boleh memburukkan lagi prestasi akibat ketidaksempurnaan dalam nadi itu sendiri.

Rajah di bawah menggambarkan dynamical decoupling dengan urutan nadi XX. Litar abstrak di sebelah kiri dipetakan ke jadual nadi gelombang mikro di kanan atas. Kanan bawah menggambarkan jadual yang sama, tetapi dengan urutan dua nadi X yang dimasukkan semasa tempoh melahu qubit pertama.

![Gambaran dynamical decoupling](../docs/images/guides/error-mitigation-explanation/dd.avif)

Dynamical decoupling boleh didayakan dengan menetapkan `enable` kepada `True` dalam [pilihan dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). Pilihan `sequence_type` boleh digunakan untuk memilih dari beberapa urutan nadi yang berbeza. Jenis urutan lalai ialah `"XX"`.

Sel kod berikut menunjukkan cara mendayakan dynamical decoupling untuk Estimator dan memilih urutan dynamical decoupling.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Pauli twirling
Twirling, juga dikenali sebagai [pengkompilan rawak](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), adalah teknik yang digunakan secara meluas untuk menukar saluran hingar sewenang-wenangnya menjadi saluran hingar dengan struktur yang lebih spesifik.

Pauli twirling adalah jenis twirling khas yang menggunakan operasi Pauli. Ia mempunyai kesan mengubah sebarang saluran kuantum menjadi saluran Pauli. Apabila dilakukan sahaja, ia boleh mengurangkan hingar koheren kerana hingar koheren cenderung terkumpul secara kuadratik dengan bilangan operasi, sedangkan hingar Pauli terkumpul secara linear. Pauli twirling sering digabungkan dengan teknik mitigasi ralat lain yang berfungsi lebih baik dengan hingar Pauli berbanding hingar sewenang-wenangnya.

Pauli twirling dilaksanakan dengan mengapit set gate yang dipilih dengan gate Pauli satu qubit yang dipilih secara rawak sedemikian rupa sehingga kesan ideal gate tetap sama. Hasilnya ialah satu litar digantikan dengan ensemble rawak litar, semua dengan kesan ideal yang sama. Apabila mensampling litar, sampel diambil dari pelbagai contoh rawak, bukan hanya satu.

![Gambaran Pauli twirling](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Oleh kerana sebahagian besar ralat dalam perkakasan kuantum semasa datang dari gate dua qubit, teknik ini sering diterapkan secara eksklusif pada gate dua qubit (asli). Rajah berikut menggambarkan beberapa Pauli twirl untuk gate CNOT dan ECR. Setiap litar dalam satu baris mempunyai kesan ideal yang sama.

![Gambaran twirl gate](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Pauli twirling boleh didayakan dengan menetapkan `enable_gates` kepada `True` dalam [pilihan twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Pilihan penting lain termasuk:

- `num_randomizations`: Bilangan contoh litar yang hendak diambil dari ensemble litar yang di-twirl.
- `shots_per_randomization`: Bilangan shot yang hendak disampling dari setiap contoh litar.

Sel kod berikut menunjukkan cara mendayakan Pauli twirling dan menetapkan pilihan ini untuk estimator. Tiada satu pun pilihan ini perlu ditetapkan secara eksplisit.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Pemadaman ralat readout yang di-twirl (TREX)
[Pemadaman ralat readout yang di-twirl (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) mengurangkan kesan ralat pengukuran untuk anggaran nilai jangkaan boleh cerap Pauli.
 Ia berdasarkan konsep pengukuran yang di-twirl, yang dicapai dengan menggantikan gate pengukuran secara rawak dengan urutan (1) gate Pauli X, (2) pengukuran, dan (3) pembalikkan bit klasik. Sama seperti dalam twirling gate standard, urutan ini setara dengan pengukuran biasa tanpa adanya hingar, seperti yang digambarkan dalam rajah berikut:

![Gambaran pengukuran twirling](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

Dalam kehadiran ralat readout, pengukuran twirling mempunyai kesan mendiagonalkan matriks pemindahan ralat readout, menjadikannya mudah untuk disongsangkan. Menganggar matriks pemindahan ralat readout memerlukan melaksanakan litar kalibrasi tambahan, yang memperkenalkan sedikit overhead.

TREX boleh didayakan dengan menetapkan `measure_mitigation` kepada `True` dalam [pilihan ketahanan Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) untuk Estimator. Pilihan untuk pembelajaran hingar pengukuran diterangkan [di sini](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Seperti twirling gate, anda boleh menetapkan bilangan rawak litar dan bilangan shot per rawak.

Sel kod berikut menunjukkan cara mendayakan TREX dan menetapkan pilihan ini untuk estimator. Tiada satu pun pilihan ini perlu ditetapkan secara eksplisit.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Ekstrapolasi sifar-hingar (ZNE)
Ekstrapolasi sifar-hingar (ZNE) adalah teknik untuk mengurangkan ralat dalam menganggar nilai jangkaan boleh cerap. Walaupun ia sering meningkatkan keputusan, ia tidak dijamin menghasilkan keputusan yang tidak berat sebelah.

ZNE terdiri daripada dua peringkat:

1. _Amplifikasi hingar_: Litar kuantum asal dilaksanakan beberapa kali pada kadar hingar yang berbeza.
2. _Ekstrapolasi_: Keputusan ideal dianggar dengan mengekstrapolasi keputusan nilai jangkaan berhingar ke had sifar-hingar.

Kedua-dua peringkat amplifikasi hingar dan ekstrapolasi boleh dilaksanakan dalam pelbagai cara yang berbeza. Qiskit Runtime melaksanakan amplifikasi hingar melalui "pelipatan gate digital," yang bermakna gate dua qubit digantikan dengan urutan setara gate tersebut dan inversnya. Contohnya, menggantikan uniter $U$ dengan $U U^\dagger U$ akan menghasilkan faktor amplifikasi hingar sebanyak 3. Untuk ekstrapolasi, anda boleh memilih dari salah satu daripada beberapa bentuk fungsian, termasuk padanan linear atau padanan eksponen.
Imej di bawah menggambarkan pelipatan gate digital di sebelah kiri, dan prosedur ekstrapolasi di sebelah kanan.

![Gambaran ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

ZNE boleh didayakan dengan menetapkan `zne_mitigation` kepada `True` dalam [pilihan ketahanan Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) untuk Estimator.
Pilihan Qiskit Runtime untuk ZNE diterangkan [di sini](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Pilihan berikut adalah penting:

- `noise_factors`: Faktor hingar untuk digunakan dalam amplifikasi hingar.
- `extrapolator`: Bentuk fungsian untuk digunakan dalam ekstrapolasi.

Sel kod berikut menunjukkan cara mendayakan ZNE dan menetapkan pilihan ini untuk estimator. Tiada satu pun pilihan ini perlu ditetapkan secara eksplisit.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Amplifikasi ralat kebarangkalian (PEA)
Salah satu cabaran utama dalam ZNE ialah mengamplifikasi hingar yang mempengaruhi litar sasaran secara tepat. Pelipatan gate menyediakan cara mudah untuk melakukan amplifikasi ini, tetapi berpotensi tidak tepat dan mungkin membawa kepada keputusan yang salah. Lihat artikel ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), dan khususnya halaman 4 maklumat tambahan untuk butiran. Amplifikasi ralat kebarangkalian menyediakan pendekatan yang lebih tepat untuk amplifikasi ralat melalui pembelajaran hingar.

PEA adalah teknik yang lebih canggih yang melakukan eksperimen awal untuk merekonstruksi hingar dan kemudian menggunakan maklumat ini untuk melakukan amplifikasi yang tepat. Ia bermula dengan mempelajari model hingar yang di-twirl setiap lapisan gate mencirikan dalam litar sebelum dijalankan (lihat [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) untuk pilihan pembelajaran yang relevan). Selepas fasa pembelajaran, litar dilaksanakan pada setiap faktor hingar, di mana setiap lapisan mencirikan litar diamplifikasi dengan menyuntik hingar satu qubit secara kebarangkalian berkadar dengan model hingar yang dipelajari. Lihat artikel ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) untuk butiran lanjut.

PEA terdiri daripada tiga peringkat:
1. _Pembelajaran_: Model hingar yang di-twirl setiap lapisan gate mencirikan dalam litar dipelajari.
1. _Amplifikasi hingar_: Litar kuantum asal dilaksanakan beberapa kali pada faktor hingar yang berbeza.
2. _Ekstrapolasi_: Keputusan ideal dianggar dengan mengekstrapolasi keputusan nilai jangkaan berhingar ke had sifar-hingar.

Untuk eksperimen skala utiliti, PEA sering menjadi pilihan terbaik.

Oleh kerana PEA adalah teknik amplifikasi hingar ZNE, anda juga perlu mendayakan ZNE dengan menetapkan `resilience.zne_mitigation = True`. Pilihan [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) lain juga boleh digunakan untuk menetapkan ekstrapolator, tahap amplifikasi, dan sebagainya. PEA memerlukan model hingar, yang dijana secara automatik apabila menggunakan primitif.

Coretan berikut memberikan contoh di mana PEA digunakan untuk mengurangkan keputusan kerja Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Pembatalan ralat kebarangkalian (PEC)
Pembatalan ralat kebarangkalian (PEC) adalah teknik untuk mengurangkan ralat dalam menganggar nilai jangkaan boleh cerap. Tidak seperti ZNE, ia mengembalikan anggaran yang tidak berat sebelah bagi nilai jangkaan. Namun, ia umumnya menanggung overhead yang lebih besar.

Dalam PEC, kesan litar sasaran ideal dinyatakan sebagai gabungan linear litar berhingar yang sebenarnya boleh dilaksanakan dalam amalan:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

Output litar ideal kemudian boleh dihasilkan semula dengan melaksanakan contoh litar berhingar yang berbeza yang diambil dari ensemble rawak yang ditakrifkan oleh gabungan linear. Jika pekali $\eta_i$ membentuk taburan kebarangkalian, ia boleh digunakan terus sebagai kebarangkalian ensemble. Dalam amalan, sesetengah pekali adalah negatif, jadi ia membentuk taburan quasi-kebarangkalian. Ia masih boleh digunakan untuk menakrifkan ensemble rawak, tetapi terdapat overhead pensampelan yang berkaitan dengan kenegatifan taburan quasi-kebarangkalian, yang dicirikan oleh kuantiti

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

Overhead pensampelan adalah faktor pendarab pada bilangan shot yang diperlukan untuk menganggar nilai jangkaan kepada ketepatan tertentu, berbanding bilangan shot yang diperlukan dari litar ideal. Ia berskala secara kuadratik dengan $\gamma$, yang seterusnya berskala secara eksponen dengan kedalaman litar.

PEC boleh didayakan dengan menetapkan `pec_mitigation` kepada `True` dalam [pilihan ketahanan Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) untuk Estimator.
Pilihan Qiskit Runtime untuk PEC diterangkan [di sini](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Had pada overhead pensampelan boleh ditetapkan menggunakan pilihan `max_overhead`. Perlu diingat bahawa mengehadkan overhead pensampelan boleh menyebabkan ketepatan keputusan melebihi ketepatan yang diminta. Nilai lalai `max_overhead` ialah 100.

Sel kod berikut menunjukkan cara mendayakan PEC dan menetapkan pilihan `max_overhead` untuk estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Langkah seterusnya
- Lihat [tutorial](/tutorials/combine-error-mitigation-techniques) tentang menggabungkan pilihan mitigasi ralat dengan primitif Estimator.
- [Konfigurasi mitigasi ralat.](configure-error-mitigation)
- [Konfigurasi penindasan ralat.](configure-error-suppression)
- Terokai [pilihan](runtime-options-overview) lain untuk primitif Qiskit Runtime.
- Putuskan [mod pelaksanaan](execution-modes) mana yang hendak digunakan untuk menjalankan kerja anda.